# Computing Pareto front using Optuna

In [1]:
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import plotly.express as px
import plotly.graph_objects as go

In [2]:
import warnings
warnings.simplefilter('ignore')

# Load the models trained

In [3]:
# load the models trained

with open('./data_common/best_RF_E.pickle', mode='rb') as f:
    rf_E = pickle.load(f)

In [4]:
with open('./data_common/best_RF_density.pickle', mode='rb') as f:
    rf_density = pickle.load(f)

In [5]:
with open('./data_common/best_RF_Tg.pickle', mode='rb') as f:
    rf_Tg = pickle.load(f)

## Try search in wide area

In [6]:
# define objective functions

def objective(trial):
    # define search ranges for each composition
    sio2 = trial.suggest_float("SiO2", 40.0, 85.0)
    b2o3 = trial.suggest_float("B2O3", 0.0, 40.0)
    al2o3 = trial.suggest_float("Al2O3", 5.0, 20.0)
    mgo = trial.suggest_float("MgO", 0.0, 15.0)
    cao = trial.suggest_float("CaO", 0.0, 15.0)
    na2o = trial.suggest_float("Na2O", 0.0, 40.0)
    k2o = trial.suggest_float("K2O", 0.0, 15.0)
    
    # adjust the composition amount to 100%    
    raw_composition = np.array([sio2, b2o3, al2o3, mgo, cao, na2o, k2o])

    raw_sum = np.sum(raw_composition)
    if raw_sum > 0:
        composition = 100.0 * raw_composition / raw_sum
    else:
        # if all composition = 0.0
        composition = np.full_like(raw_composition, 100.0 / len(raw_composition))
    
    # adjust shape to (1, n_features)
    X_input = composition.reshape(1, -1)
    
    # predict properties
    e_pred = rf_E.predict(X_input)[0]
    density_pred = rf_density.predict(X_input)[0]
    tg_pred = rf_Tg.predict(X_input)[0]
    
    return e_pred, density_pred, tg_pred

In [7]:
# determine optimazion conditions : E(maximize), density(maximize), Tg(minimize)

study = optuna.create_study(directions=["maximize", "maximize", "minimize"])

[I 2026-09-26 09:55:55,301] A new study created in memory with name: no-name-ed63e845-8740-4562-887b-a5f65eec5738


In [8]:
# profibit showing log

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [9]:
# optimization process

study.optimize(objective, n_trials=100)

In [10]:
# print results

pareto_front = study.best_trials 

print(f"nunber of Pareto fronts = {len(pareto_front)}")

nunber of Pareto fronts = 19


In [11]:
# make table of the results

df_pareto_opt_comp = pd.DataFrame()

for trial in pareto_front:
    params = np.array(list(trial.params.values()))
    normalized_comp = pd.DataFrame(100.0 * params / np.sum(params)).T
    comp_target = pd.concat([normalized_comp, pd.DataFrame(trial.values).T], axis = 1)    
    df_pareto_opt_comp = pd.concat([df_pareto_opt_comp, comp_target])

df_pareto_opt_comp.columns = ["SiO2", "B2O3", "Al2O3", "MgO", "CaO", "Na2O", "K2O", "E", "density", "Tg"]
print(df_pareto_opt_comp.shape)
df_pareto_opt_comp.head()

(19, 10)


,SiO2,B2O3,Al2O3,MgO,CaO,Na2O,K2O,E,density,Tg
0,41.440757,11.734363,5.216662,5.803975,0.719589,28.845995,6.238658,72.645411,2.486452,507.342177
0,49.158658,2.860882,5.180559,4.227574,9.819404,19.272242,9.480680,76.278160,2.523013,545.215136
0,44.086752,15.624726,3.245355,5.587863,3.106885,21.482103,6.866316,76.593286,2.485933,532.038824
0,46.762767,16.994094,3.672773,4.062687,2.156129,20.197456,6.154094,77.471941,2.487995,541.314468
0,39.366658,16.439178,5.366273,5.665309,2.612561,28.381016,2.169005,73.183172,2.484924,518.920493


In [12]:
# visualization

all_values = np.array([t.values for t in study.trials])           # targets from all study
pareto_values = np.array([t.values for t in study.best_trials])   # targets optimized

df_all_values = pd.DataFrame(all_values, columns = ['E', 'density', 'Tg'])        # targets from all study
df_pareto_values = pd.DataFrame(pareto_values, columns = ['E', 'density', 'Tg'])  # targets optimized
df_all_values['group'] = 'All'                                                    # add groups
df_pareto_values['group'] = 'Pareto'
df = pd.concat([df_all_values, df_pareto_values])

fig = px.scatter_3d(df, x='E', y='density', z='Tg', color='group', width=800, height=500)   # 3d plot
fig.update_layout(margin = dict(l=0, r=0, b=20, t=20))       # margins to depict
fig.update_traces(marker = dict(size = 3))
fig.show()

## Save the final results for comparison

In [13]:
# save the dataframe

df_pareto_opt_comp.to_csv('./data_common/df_pareto_opt_comp_def.csv', index = False)